<a href="https://colab.research.google.com/github/Muskan-Sh-23/Machine-learning/blob/main/Random_Forest/Random_Forest_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Random Forest from Scratch

In [28]:
import numpy as np
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

X = housing.data
y = housing.target

feature_names = housing.feature_names

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", feature_names)
print("First 5 targets:", y[:5])

X shape: (20640, 8)
y shape: (20640,)
Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
First 5 targets: [4.526 3.585 3.521 3.413 3.422]


In [29]:
np.random.seed(42)

indices = np.random.permutation(len(X))

test_size = int(0.2 * len(X))

test_indices = indices[:test_size]
train_indices = indices[test_size:]

X_train = X[train_indices]
X_test = X[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 16512
Testing samples: 4128


In [30]:
def mse(y):
    if len(y) == 0:
        return 0.0

    mean_value = np.mean(y)
    squared_errors = (y - mean_value) ** 2

    return np.mean(squared_errors)

In [31]:
def split_score(X, y, feature_idx, threshold):
    left_mask = X[:, feature_idx] <= threshold
    right_mask = ~left_mask

    y_left = y[left_mask]
    y_right = y[right_mask]

    # Reject splits that create an empty child
    if len(y_left) == 0 or len(y_right) == 0:
        return float("inf")

    n = len(y)

    weighted_mse = (
        (len(y_left) / n) * mse(y_left)
        + (len(y_right) / n) * mse(y_right)
    )

    return weighted_mse

In [32]:
def find_best_split(X, y):
    best_score = float("inf")
    best_feature = None
    best_threshold = None

    n_features = X.shape[1]

    for feature_idx in range(n_features):
        values = np.unique(X[:, feature_idx])
        values.sort()

        thresholds = (values[:-1] + values[1:]) / 2

        for threshold in thresholds:
            score = split_score(
                X, y, feature_idx, threshold
            )

            if score < best_score:
                best_score = score
                best_feature = feature_idx
                best_threshold = threshold

    return best_feature, best_threshold, best_score

In [33]:
def build_tree(X, y, depth=0, max_depth=10, min_samples_split=2):
    # Stopping condition
    if depth >= max_depth or len(y) < min_samples_split or len(y) <= 1:
        return {"value": np.mean(y)}

    feature, threshold, score = find_best_split(X, y)

    # Stop if no valid split exists
    if feature is None:
        return {"value": np.mean(y)}

    left_mask = X[:, feature] <= threshold
    right_mask = ~left_mask

    # Avoid empty children
    if not np.any(left_mask) or not np.any(right_mask):
        return {"value": np.mean(y)}

    # Recursively build both subtrees
    left_subtree = build_tree(
        X[left_mask], y[left_mask],
        depth + 1, max_depth, min_samples_split
    )

    right_subtree = build_tree(
        X[right_mask], y[right_mask],
        depth + 1, max_depth, min_samples_split
    )

    return {
        "feature": feature,
        "threshold": threshold,
        "left": left_subtree,
        "right": right_subtree
    }

In [34]:
def predict_one(x, tree):
    # Leaf node
    if "value" in tree:
        return tree["value"]

    feature = tree["feature"]
    threshold = tree["threshold"]

    if x[feature] <= threshold:
        return predict_one(x, tree["left"])
    else:
        return predict_one(x, tree["right"])


def predict_tree(X, tree):
    return np.array([predict_one(x, tree) for x in X])

In [35]:
def find_best_split(X, y, feature_indices):
    best_score = float("inf")
    best_feature = None
    best_threshold = None

    for feature_idx in feature_indices:
        values = np.unique(X[:, feature_idx])
        values.sort()

        thresholds = (values[:-1] + values[1:]) / 2

        for threshold in thresholds:
            score = split_score(X, y, feature_idx, threshold)

            if score < best_score:
                best_score = score
                best_feature = feature_idx
                best_threshold = threshold

    return best_feature, best_threshold, best_score

In [36]:
def build_tree(
    X, y, depth=0, max_depth=10,
    min_samples_split=2, max_features=None
):
    n_samples, n_features = X.shape

    if (
        depth >= max_depth
        or n_samples < min_samples_split
        or n_samples <= 1
    ):
        return {"value": np.mean(y)}

    if max_features is None:
        max_features = max(1, int(np.sqrt(n_features)))

    feature_indices = np.random.choice(
        n_features,
        size=min(max_features, n_features),
        replace=False
    )

    feature, threshold, score = find_best_split(
        X, y, feature_indices
    )

    if feature is None:
        return {"value": np.mean(y)}

    left_mask = X[:, feature] <= threshold
    right_mask = ~left_mask

    if not np.any(left_mask) or not np.any(right_mask):
        return {"value": np.mean(y)}

    return {
        "feature": feature,
        "threshold": threshold,
        "left": build_tree(
            X[left_mask], y[left_mask],
            depth + 1, max_depth,
            min_samples_split, max_features
        ),
        "right": build_tree(
            X[right_mask], y[right_mask],
            depth + 1, max_depth,
            min_samples_split, max_features
        )
    }

In [37]:
def bootstrap_sample(X, y):
    n_samples = len(X)

    indices = np.random.choice(
        n_samples,
        size=n_samples,
        replace=True
    )

    return X[indices], y[indices]

In [38]:
class RandomForestRegressorScratch:
    def __init__(
        self,
        n_estimators=10,
        max_depth=10,
        min_samples_split=2,
        max_features=None,
        random_state=None
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        if self.random_state is not None:
            np.random.seed(self.random_state)

        self.trees = []

        for _ in range(self.n_estimators):
            X_sample, y_sample = bootstrap_sample(X, y)

            tree = build_tree(
                X_sample,
                y_sample,
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features
            )

            self.trees.append(tree)

        return self

    def predict(self, X):
        predictions = np.array([
            predict_tree(X, tree)
            for tree in self.trees
        ])

        return np.mean(predictions, axis=0)

In [39]:
def mean_squared_error_scratch(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


def r2_score_scratch(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    if ss_tot == 0:
        return 0.0

    return 1 - (ss_res / ss_tot)

In [40]:
model = RandomForestRegressorScratch(
    n_estimators=3,
    max_depth=8,
    min_samples_split=5,
    max_features=3,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Number of trees:", len(model.trees))
print("Test MSE:", mean_squared_error_scratch(y_test, y_pred))
print("Test R²:", r2_score_scratch(y_test, y_pred))

Number of trees: 3
Test MSE: 0.38072476245210485
Test R²: 0.7094611157478546
